In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import datetime as dt
import os
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============= НАСТРОЙКИ =============
saber_base_dir = 'G:/SABER_L2A'
matching_dir = 'C:/Users/Maks/Desktop/Jupyter/output_full_saber_data'
years = [2012, 2013, 2014, 2015, 2016, 2017, 2018]

n_profiles_before = 10  # количество профилей ДО события
n_profiles_after = 10   # количество профилей ПОСЛЕ события

# Цветовая карта
colors = [(0.5, 0.0, 0.5), (0, 0, 1), (0, 0.5, 1), (0.5, 1, 0.5),
          (1, 1, 0), (1, 0.5, 0), (1, 0, 0), (0.5, 0, 0)]
custom_cmap = LinearSegmentedColormap.from_list("custom_coolwarm", colors, N=100)

# Создаем папку для результатов
output_dir = 'C:/Users/Maks/Desktop/Jupyter/super_composite'
os.makedirs(output_dir, exist_ok=True)

print("=" * 80)
print("СУПЕР-КОМПОЗИТ ТЕМПЕРАТУРНЫХ ПРОФИЛЕЙ")
print(f"Окно: ±{n_profiles_before}/{n_profiles_after} профилей")
print("=" * 80)

# ============= ФУНКЦИЯ ДЛЯ ЗАГРУЗКИ ПРОФИЛЕЙ =============
def load_saber_profiles_organized(nc_file_path):
    try:
        ds = xr.open_dataset(nc_file_path, decode_timedelta=True)
        
        date_value = float(ds['date'].values[0])
        date_str = str(int(date_value))
        year_val = int(date_str[:4])
        day_of_year = int(date_str[4:])
        base_date = dt.datetime(year_val, 1, 1) + dt.timedelta(days=day_of_year - 1)
        
        profiles = []
        
        for profile_idx in range(len(ds['ktemp'])):
            temps = ds['ktemp'].values[profile_idx]
            alts = ds['tpaltitude'].values[profile_idx]
            times = ds['time'].values[profile_idx]
            
            valid_mask = ~np.isnan(temps)
            if not valid_mask.any():
                continue
            
            valid_indices = np.where(valid_mask)[0]
            
            first_valid_idx = valid_indices[0]
            time_ms = float(times[first_valid_idx])
            profile_time = base_date + dt.timedelta(milliseconds=time_ms)
            profile_time = pd.Timestamp(profile_time).floor('s')
            
            profile_data = []
            for idx in valid_indices:
                profile_data.append({
                    'altitude_km': float(alts[idx]),
                    'temperature_K': float(temps[idx])
                })
            
            profiles.append({
                'profile_index': profile_idx,
                'datetime': profile_time,
                'data': pd.DataFrame(profile_data)
            })
        
        ds.close()
        return profiles
    
    except Exception as e:
        return None

# ============= ОСНОВНОЙ ЦИКЛ СБОРА ДАННЫХ =============
print("\nСБОР ДАННЫХ ПО ВСЕМ КЛАСТЕРАМ")

all_points = []
processed_events = 0
total_profiles = 0

for year in tqdm(years, desc="Годы"):
    matching_path = os.path.join(matching_dir, f'{year}_saber_overpass_matching.h5')
    
    if not os.path.exists(matching_path):
        continue
    
    try:
        matches = pd.read_hdf(matching_path, key='matches')
    except Exception:
        continue
    
    for idx, row in matches.iterrows():
        # Получаем данные
        if isinstance(row['satellite_file_name'], pd.Series):
            saber_filename = row['satellite_file_name'].iloc[0].replace('.nc', '')
        else:
            saber_filename = row['satellite_file_name'].replace('.nc', '')
        
        if isinstance(row['matched_time'], pd.Series):
            matched_time = pd.to_datetime(row['matched_time'].iloc[0])
        else:
            matched_time = pd.to_datetime(row['matched_time'])
        
        if isinstance(idx, pd.Series):
            cluster_id = idx.iloc[0]
        else:
            cluster_id = idx
        
        file_parts = saber_filename.split('_')
        yyddd = file_parts[2]
        day = yyddd[-3:]
        saber_path = os.path.join(saber_base_dir, str(year), day, f"{saber_filename}.nc")
        
        if not os.path.exists(saber_path):
            continue
        
        # Загружаем профили
        profiles = load_saber_profiles_organized(saber_path)
        
        if profiles is None or len(profiles) == 0:
            continue
        
        # Находим ближайший профиль к событию
        time_diffs = [abs(p['datetime'] - matched_time).total_seconds() for p in profiles]
        closest_idx = np.argmin(time_diffs)
        
        # Берем ±10 профилей
        start_idx = max(0, closest_idx - n_profiles_before)
        end_idx = min(len(profiles), closest_idx + n_profiles_after + 1)
        profiles_in_window = profiles[start_idx:end_idx]
        
        total_profiles += len(profiles_in_window)
        
        # Сохраняем все точки с временем, округленным до минут
        for p in profiles_in_window:
            time_offset_seconds = (p['datetime'] - matched_time).total_seconds()
            time_offset_minutes = time_offset_seconds / 60
            
            # Округление до минут: 0-30 сек → 0, 30-90 сек → 1, и т.д.
            time_offset_rounded = int(np.floor(time_offset_minutes + 0.5))
            
            # Ограничиваем диапазон от -10 до +10
            if time_offset_rounded < -10:
                time_offset_rounded = -10
            if time_offset_rounded > 10:
                time_offset_rounded = 10
            
            for _, row_data in p['data'].iterrows():
                all_points.append({
                    'time_offset_minutes': time_offset_rounded,
                    'altitude_km': row_data['altitude_km'],
                    'temperature_K': row_data['temperature_K']
                })
        
        processed_events += 1

# ============= АГРЕГАЦИЯ ДАННЫХ =============
print(f"\nОбработано событий: {processed_events}")
print(f"Всего профилей: {total_profiles}")
print(f"Всего сырых точек: {len(all_points):,}")

if len(all_points) == 0:
    print("Нет данных для построения графиков!")
else:
    df_points = pd.DataFrame(all_points)
    
    # Округляем высоту до целых км для группировки
    df_points['altitude_rounded'] = df_points['altitude_km'].round()
    
    # Группируем по времени и высоте
    print("\nАГРЕГАЦИЯ ДАННЫХ ПО ГРУППАМ (время, высота)")
    
    grouped = df_points.groupby(['time_offset_minutes', 'altitude_rounded'])['temperature_K'].agg(['mean', 'median', 'count']).reset_index()
    grouped.columns = ['time_offset_minutes', 'altitude_km', 'temperature_mean', 'temperature_median', 'count']
    
    print(f"Сформировано групп: {len(grouped)}")
    print(f"Диапазон времени: {grouped['time_offset_minutes'].min()} - {grouped['time_offset_minutes'].max()} мин")
    print(f"Диапазон высот: {grouped['altitude_km'].min()} - {grouped['altitude_km'].max()} км")
    
    # Сохраняем агрегированные данные
    grouped.to_csv(os.path.join(output_dir, 'aggregated_data.csv'), index=False)
    
    # ============= 1. ГРАФИК: МЕДИАНА (contourf) =============
    print("\n1. Построение графика медианных температур (contourf)...")
    
    # Создаем сетку для contourf
    time_values = np.arange(-10, 11, 1)
    alt_values = np.arange(0, 121, 1)
    
    median_matrix = np.full((len(alt_values), len(time_values)), np.nan)
    
    for i, alt in enumerate(alt_values):
        for j, t in enumerate(time_values):
            val = grouped[(grouped['altitude_km'] == alt) & (grouped['time_offset_minutes'] == t)]['temperature_median'].values
            if len(val) > 0:
                median_matrix[i, j] = val[0]
    
    fig, ax = plt.subplots(figsize=(16, 10))
    
    im = ax.contourf(time_values, alt_values, median_matrix, 
                     levels=50, cmap=custom_cmap, vmin=180, vmax=260)
    
    ax.axvline(x=0, color='black', linestyle='--', linewidth=2, alpha=0.8,
               label='Момент пролета спутника')
    
    ax.set_xlabel('Время относительно попадания (минуты)', fontsize=14)
    ax.set_ylabel('Высота (км)', fontsize=14)
    ax.set_title(f'Медианная температура атмосферы\n'
                 f'N = {processed_events} событий, ±{n_profiles_before}/{n_profiles_after} профилей',
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(time_values)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Температура (K)', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'super_composite_median.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # ============= 2. ГРАФИК: СРЕДНЕЕ (contourf) =============
    print("2. Построение графика средних температур (contourf)...")
    
    mean_matrix = np.full((len(alt_values), len(time_values)), np.nan)
    
    for i, alt in enumerate(alt_values):
        for j, t in enumerate(time_values):
            val = grouped[(grouped['altitude_km'] == alt) & (grouped['time_offset_minutes'] == t)]['temperature_mean'].values
            if len(val) > 0:
                mean_matrix[i, j] = val[0]
    
    fig, ax = plt.subplots(figsize=(16, 10))
    
    im = ax.contourf(time_values, alt_values, mean_matrix, 
                     levels=50, cmap=custom_cmap, vmin=180, vmax=260)
    
    ax.axvline(x=0, color='black', linestyle='--', linewidth=2, alpha=0.8,
               label='Момент пролета спутника')
    
    ax.set_xlabel('Время относительно попадания (минуты)', fontsize=14)
    ax.set_ylabel('Высота (км)', fontsize=14)
    ax.set_title(f'Средняя температура атмосферы\n'
                 f'N = {processed_events} событий, ±{n_profiles_before}/{n_profiles_after} профилей',
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(time_values)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Температура (K)', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'super_composite_mean.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # ============= 3. ГРАФИК: SCATTER PLOT ПО МЕДИАННЫМ ЗНАЧЕНИЯМ =============
    print("3. Построение scatter plot по медианным значениям...")
    
    fig, ax = plt.subplots(figsize=(20, 12))
    
    grouped_filtered = grouped[grouped['count'] >= 5]  
    
    scatter = ax.scatter(grouped_filtered['time_offset_minutes'], 
                        grouped_filtered['altitude_km'],
                        c=grouped_filtered['temperature_median'], 
                        s=200, alpha=0.7, cmap=custom_cmap,
                        vmin=130, vmax=300, linewidth=0.3)
    
    ax.axvline(x=0, color='black', linestyle='--', linewidth=2, alpha=0.5,
               label='Момент пролета спутника')
    
    ax.set_xlabel('Время относительно попадания (минуты)', fontsize=34)
    ax.set_ylabel('Высота (км)', fontsize=34)
    ax.set_title(f'Медианная температура ({processed_events} событий)',
                 fontsize=40, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 120)
    ax.set_xlim(-10.5, 10.5)
    ax.set_xticks(np.arange(-10, 11, 1))
    ax.tick_params(axis='x', labelsize=26)
    ax.tick_params(axis='y', labelsize=30)
    
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Температура (K)', fontsize=30)
    cbar.ax.tick_params(labelsize=30)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'super_composite_median_scatter.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # ============= 4. ГРАФИК: SCATTER PLOT ПО СРЕДНИМ ЗНАЧЕНИЯМ =============
    print("4. Построение scatter plot по средним значениям...")
    
    fig, ax = plt.subplots(figsize=(20, 12))
    
    scatter = ax.scatter(grouped_filtered['time_offset_minutes'], 
                        grouped_filtered['altitude_km'],
                        c=grouped_filtered['temperature_mean'], 
                        s=200, alpha=0.7, cmap=custom_cmap,
                        vmin=130, vmax=300, linewidth=0.3)
    
    ax.axvline(x=0, color='black', linestyle='--', linewidth=2, alpha=0.5,
               label='Момент пролета спутника')
    
    ax.set_xlabel('Время относительно попадания (минуты)', fontsize=34)
    ax.set_ylabel('Высота (км)', fontsize=34)
    ax.set_title(f'Средняя температура ({processed_events} событий)',
                 fontsize=40, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 120)
    ax.set_xlim(-10.5, 10.5)
    ax.set_xticks(np.arange(-10, 11, 1))
    ax.tick_params(axis='x', labelsize=26)
    ax.tick_params(axis='y', labelsize=30)
    
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Температура (K)', fontsize=30)
    cbar.ax.tick_params(labelsize=30)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'super_composite_mean_scatter.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # ============= 5. ГРАФИК: СРАВНЕНИЕ ПРОФИЛЕЙ =============
    print("5. Построение графика сравнения профилей...")
    
    alt_profile_values = np.arange(0, 121, 1)
    
    fon_profiles = []
    for alt in alt_profile_values:
        vals = grouped[(grouped['altitude_km'] == alt) & 
                       (grouped['time_offset_minutes'] >= -10) & 
                       (grouped['time_offset_minutes'] <= -6)]['temperature_median'].values
        if len(vals) > 0:
            fon_profiles.append(np.median(vals))
        else:
            fon_profiles.append(np.nan)
    
    event_profiles = []
    for alt in alt_profile_values:
        vals = grouped[(grouped['altitude_km'] == alt) & 
                       (grouped['time_offset_minutes'] == 0)]['temperature_median'].values
        if len(vals) > 0:
            event_profiles.append(vals[0])
        else:
            event_profiles.append(np.nan)
    
    after_profiles = []
    for alt in alt_profile_values:
        vals = grouped[(grouped['altitude_km'] == alt) & 
                       (grouped['time_offset_minutes'] >= 5) & 
                       (grouped['time_offset_minutes'] <= 9)]['temperature_median'].values
        if len(vals) > 0:
            after_profiles.append(np.median(vals))
        else:
            after_profiles.append(np.nan)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    valid_idx = ~np.isnan(fon_profiles) & ~np.isnan(event_profiles)
    alt_valid = np.array(alt_profile_values)[valid_idx]
    
    ax.plot(np.array(fon_profiles)[valid_idx], alt_valid, 'b-', linewidth=2, 
            label=f'Фон (t = -10...-6 мин)')
    ax.plot(np.array(event_profiles)[valid_idx], alt_valid, 'r-', linewidth=2.5, 
            label=f'Момент пролета (t = 0 мин)')
    
    valid_idx2 = ~np.isnan(after_profiles)
    alt_valid2 = np.array(alt_profile_values)[valid_idx2]
    ax.plot(np.array(after_profiles)[valid_idx2], alt_valid2, 'g--', linewidth=2, 
            label=f'После (t = +5...+9 мин)')
    
    ax.set_xlabel('Температура (K)', fontsize=14)
    ax.set_ylabel('Высота (км)', fontsize=14)
    ax.set_title(f'Сравнение медианных температурных профилей\n'
                 f'N = {processed_events} событий',
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 120)
    ax.legend(loc='best')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'profile_comparison.png'), dpi=300, bbox_inches='tight')
    plt.close()

# ============= ФИНАЛЬНЫЙ ВЫВОД =============
print("\n" + "=" * 80)
print("РЕЗУЛЬТАТЫ СОХРАНЕНЫ")
print("=" * 80)
print(f"Папка: {output_dir}")
print("\nСозданные файлы:")
print("  1. aggregated_data.csv - агрегированные данные (среднее, медиана, кол-во)")
print("  2. super_composite_median.png - Супер-композит (медиана, contourf)")
print("  3. super_composite_mean.png - Супер-композит (среднее, contourf)")
print("  4. super_composite_median_scatter.png - Scatter plot по медианным значениям")
print("  5. super_composite_mean_scatter.png - Scatter plot по средним значениям")
print("  6. profile_comparison.png - Сравнение профилей")
print("=" * 80)